In [ ]:
!pip install -q yfinance pandas

import yfinance as yf
import pandas as pd

# 1. Select US stocks
stocks = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "NVDA",
    "META", "TSLA", "NFLX", "AMD", "INTC",
    "JPM", "V", "WMT", "COST", "DIS",
    "ORCL", "CRM", "ADBE", "AVGO", "KO"
]

# 2. Download historical stock prices
data = yf.download(
    stocks,
    period="6mo",
    interval="1d",
    auto_adjust=True,
    progress=False
)

prices = data["Close"].dropna(axis=1, how="all")

print("Successfully downloaded:", len(prices.columns), "stocks")

# 3. Calculate stock performance
results = []

for stock in prices.columns:
    stock_data = prices[stock].dropna()

    if len(stock_data) < 30:
        continue

    momentum = (
        stock_data.iloc[-1] / stock_data.iloc[-20] - 1
    ) * 100

    total_return = (
        stock_data.iloc[-1] / stock_data.iloc[0] - 1
    ) * 100

    volatility = stock_data.pct_change().std() * (252 ** 0.5) * 100

    results.append({
        "Stock": stock,
        "Current Price ($)": round(float(stock_data.iloc[-1]), 2),
        "20-Day Momentum (%)": round(float(momentum), 2),
        "6-Month Return (%)": round(float(total_return), 2),
        "Volatility (%)": round(float(volatility), 2)
    })

# 4. Create recommendation ranking
recommendations = pd.DataFrame(results)

if not recommendations.empty:

    recommendations["Score"] = (
        recommendations["20-Day Momentum (%)"] * 0.4
        + recommendations["6-Month Return (%)"] * 0.6
        - recommendations["Volatility (%)"] * 0.2
    )

    recommendations = recommendations.sort_values(
        by="Score",
        ascending=False
    )

    print("\nTop 5 Recommended Stocks:")
    display(recommendations.head(5))

else:
    print("No stock data available. Please check your internet connection.")

In [ ]:

import os
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# Take top 3 stocks from your actual results
sample_data = recommendations.head(3).to_string(index=False)

SYSTEM_PROMPT = """
You are a professional US stock market research analyst.
Analyze stocks using historical returns, momentum, volatility,
and quantitative scores.

Be concise, objective, and practical.
Never guarantee profits or invent missing financial information.
"""

USER_PROMPT = f"""
Analyze these top 3 stocks using the actual downloaded data:

{sample_data}

Provide:
1. Best candidate and why.
2. Comparison of performance and risk.
3. Main concern for each stock.
4. Final recommendation.

Keep the analysis under 150 words.
Remember that historical performance does not guarantee future returns.
"""

response = client.responses.create(
    model="gpt-6-luna",
    reasoning_effort="none",
    instructions=SYSTEM_PROMPT,
    input=USER_PROMPT
)

print(response.output_text)